In [ ]:
import os
import time
import base64
import pathlib
import pandas as pd
import json
import re
import hashlib
from datetime import datetime, timezone
from collections import defaultdict
from openai import OpenAI
from google.colab import drive
from google.colab import userdata

# ==========================================
# 1. SETUP & API KEY
# ==========================================
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

try:
    os.environ["RadLE_API"] = userdata.get('RadLE_API')
except Exception:
    print("❌ ERROR: Could not find RadLE_API in Colab Secrets.")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["RadLE_API"],
)

In [ ]:

# ==========================================
# 2. RadLE v2 FRONTIER MODELS (THE FULL 10)
# ==========================================

# Reasoning controls are model-specific.
# Do not assume a universal internal reasoning-token budget across models.
MAX_OUTPUT_TOKENS = 16384
UNIVERSAL_TEMPERATURE = 0.01

NO_TEMPERATURE_MODELS = {
    "openai/gpt-5.5",
    "anthropic/claude-opus-4.7",
}

MODELS = [
    # --- Frontier Closed ---

    {
        "name": "gpt_5_5",
        "id": "openai/gpt-5.5",
        "extra": {"reasoning": {"effort": "xhigh"}},
    },

    {
        "name": "claude_4_7_opus",
        "id": "anthropic/claude-opus-4.7",
        "extra": {
            "reasoning": {"enabled": True},
            "verbosity": "max",
        },
    },

    {
        "name": "gemini_3_1_pro",
        "id": "google/gemini-3.1-pro-preview",
        "extra": {"reasoning": {"effort": "high"}},
    },

    {
        "name": "grok_4_20",
        "id": "x-ai/grok-4.20",
        "extra": {"reasoning": {"effort": "xhigh"}},
    },

    {
        "name": "qwen_vl_max",
        "id": "qwen/qwen-vl-max",
        "extra": None,
    },

    # --- Frontier Open ---

    {
        "name": "gemma_4_31b",
        "id": "google/gemma-4-31b-it",
        "extra": {"reasoning": {"enabled": True}},
    },

    {
        "name": "llama_4_maverick",
        "id": "meta-llama/llama-4-maverick",
        "extra": None,
    },

    {
        "name": "pixtral_large",
        "id": "mistralai/pixtral-large-2411",
        "extra": None,
    },

    {
        "name": "glm_4_6v",
        "id": "z-ai/glm-4.6v",
        "extra": {"reasoning": {"enabled": True}},
    },

    {
        "name": "nemotron_3_omni",
        "id": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
        "extra": {"reasoning": {"enabled": True}},
    },
]

In [ ]:

# ==========================================
# 3. PROMPT
# ==========================================

PROMPT = """
System prompt:

You are a board-certified diagnostic radiologist. Given a medical image, provide the single most specific final diagnosis.

Rules:
	1.	If you can identify a diagnosis, return it. If you truly cannot, return “I don’t know”.
	2.	No abbreviations, use full words only.
	3.	No verbose descriptions of the diagnosis.
	4.	Use the following Likert confidence scale (only when a diagnosis is given) to tell how confident you are in your diagnosis:
	∙	0 = very low confidence (essentially a guess)
	∙	1 = low confidence (weak leading diagnosis, several alternatives similarly plausible)
	∙	2 = moderate confidence (one diagnosis favored, important alternatives remain)
	∙	3 = high confidence (one diagnosis clearly favored, alternatives unlikely)
	∙	4 = very high confidence (classic appearance, essentially certain)
5. If you return “I don’t know”, then return Likert score as “null”.

Final output format:  Respond with this JSON:

{"diagnosis": "<diagnosis in full words or I don't know>", "likert_score": <0-4 or null>}


Example outputs:

{"diagnosis": "Pulmonary tuberculosis", "likert_score": 3}

{"diagnosis": "Von Hippel-Lindau syndrome", "likert_score": 4}

{"diagnosis": "I don't know", "likert_score": null}
""".strip()


In [ ]:

# ==========================================
# 4. HELPER FUNCTIONS
# ==========================================

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def get_mime_type(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == '.png': return 'image/png'
    return 'image/jpeg'

def extract_json_safely(raw_text):
    """Production-grade JSON extractor that ignores reasoning blocks."""
    clean_text = re.sub(r"```json", "", raw_text, flags=re.IGNORECASE)
    clean_text = re.sub(r"```", "", clean_text)

    match = re.search(r'\{.*\}', clean_text, re.DOTALL)
    if match:
        try:
            data = json.loads(match.group(0))
            data_lower = {k.lower(): v for k, v in data.items()}
            diag = data_lower.get("diagnosis", "JSON_MISSING_KEY")
            likert = data_lower.get("likert_score", data_lower.get("likert", "NULL"))
            return diag, likert
        except json.JSONDecodeError:
            pass

    return "PARSE_FAILED", "PARSE_FAILED"


def make_json_safe(obj):
    """Convert OpenAI/Pydantic-style objects into JSON-safe structures."""
    if obj is None:
        return None
    if hasattr(obj, "model_dump"):
        return make_json_safe(obj.model_dump())
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (str, int, float, bool)):
        return obj
    return str(obj)


def is_grok_xhigh_rejection(model, error_text):
    """Detect likely rejection of reasoning.effort='xhigh' on base Grok 4.20."""
    text = str(error_text).lower()
    return (
        model["name"] == "grok_4_20"
        and "error code: 400" in text
        and (
            "reasoning" in text
            or "effort" in text
            or "unsupported" in text
            or "parameter" in text
        )
    )


In [ ]:

# ==========================================
# 5. CORE LOOP FUNCTION
# ==========================================
def run_benchmark(image_folder, output_csv, test_limit=None):
    image_paths = [str(p) for p in pathlib.Path(image_folder).rglob("*")
                   if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}]

    grouped = defaultdict(list)
    for p in image_paths:
        base_key = os.path.basename(p).split('.')[0]
        grouped[base_key].append(p)

    items = list(grouped.items())
    items.sort(key=lambda x: int(x[0]) if x[0].isdigit() else x[0])

    if test_limit:
        print(f"🧪 TEST MODE: Running on first {test_limit} cases only.")
        items = items[:test_limit]

    results = []
    print(f"🚀 Processing {len(items)} unique cases across {len(MODELS)} models...\n")

    for idx, (case_id, paths) in enumerate(items, 1):
        print(f"[{idx}/{len(items)}] Case ID: {case_id} ({len(paths)} images)")

        row = {
            "Master_Case_ID": case_id,
            "Associated_Images": ", ".join(os.path.basename(p) for p in paths),
            "Image_SHA256": ", ".join(
                hashlib.sha256(open(p, 'rb').read()).hexdigest()[:16] for p in paths
            ),
        }

        content_array = [{"type": "text", "text": PROMPT}]
        paths.sort()
        for p in paths:
            content_array.append({
                "type": "image_url",
                "image_url": {"url": f"data:{get_mime_type(p)};base64,{encode_image(p)}"}
            })

        for model in MODELS:
            print(f"  -> {model['name']}...", end="")
            api_params = None
            grok_fallback_used = False

            try:
                # UNIVERSAL RIGOR: 16k output ceiling for all models.
                # Internal reasoning budgets/effort controls remain model-specific.
                api_params = {
                    "model": model["id"],
                    "messages": [{"role": "user", "content": content_array}],
                    "max_tokens": MAX_OUTPUT_TOKENS,
                }

                if model["id"] not in NO_TEMPERATURE_MODELS:
                    api_params["temperature"] = UNIVERSAL_TEMPERATURE

                if model.get("extra"):
                    api_params["extra_body"] = model.get("extra")

                # ==========================================
                # Execute Call with Exponential Backoff
                # ==========================================
                MAX_RETRIES = 3
                response = None
                last_error = "Unknown error occurred before execution"

                for attempt in range(MAX_RETRIES):
                    try:
                        # Reset t0 on EVERY attempt so latency only measures the final successful call
                        t0 = time.time()
                        response = client.chat.completions.create(**api_params)
                        latency = round(time.time() - t0, 1)
                        break  # Success!

                    except Exception as e:
                        last_error = str(e)

                        # Special case: base Grok 4.20 may reject reasoning.effort="xhigh".
                        # Fall back once to model-page-safe reasoning.enabled=True.
                        if is_grok_xhigh_rejection(model, last_error) and not grok_fallback_used:
                            print("\n    ↪ Grok xhigh rejected; falling back to reasoning.enabled=True...")
                            api_params["extra_body"] = {"reasoning": {"enabled": True}}
                            grok_fallback_used = True
                            response = None
                            continue

                        # Hard breaks for fatal errors
                        fatal_patterns = [
                            "error code: 404",
                            "error code: 400",
                            "balance",
                            "quota",
                            "insufficient_quota",
                        ]

                        if any(p in last_error.lower() for p in fatal_patterns):
                            print(f"\n    ⚠️ FATAL ERROR (No Retry): {last_error[:100]}...")
                            break

                        if attempt < MAX_RETRIES - 1:
                            delay = 5 * (2 ** attempt)
                            print(f" \n    ⏳ Attempt {attempt + 1} failed, retrying in {delay}s...")
                            time.sleep(delay)

                # If we exhausted retries or hit a fatal break
                if response is None:
                    raise Exception(last_error)

                msg = response.choices[0].message
                raw_answer = msg.content.strip() if getattr(msg, 'content', None) else ""
                diag, likert = extract_json_safely(raw_answer)

                # Extract raw reasoning string
                raw_reasoning_text = ""
                raw_reasoning = getattr(msg, "reasoning", None)

                if raw_reasoning is not None:
                    raw_reasoning_text = str(raw_reasoning).strip()
                elif hasattr(msg, "model_extra") and msg.model_extra and "reasoning" in msg.model_extra:
                    ext_reasoning = msg.model_extra["reasoning"]
                    if ext_reasoning is not None:
                        raw_reasoning_text = str(ext_reasoning).strip()

                # Extract structured reasoning_details
                reasoning_details_obj = getattr(msg, "reasoning_details", None)

                if reasoning_details_obj is None and hasattr(msg, "model_extra") and msg.model_extra:
                    reasoning_details_obj = msg.model_extra.get("reasoning_details")

                reasoning_details_text = ""
                if reasoning_details_obj:
                    reasoning_details_text = json.dumps(
                        make_json_safe(reasoning_details_obj),
                        ensure_ascii=False,
                    )

                # Combined reasoning field for convenience
                if raw_reasoning_text and reasoning_details_text:
                    thoughts = raw_reasoning_text + "\n\n[reasoning_details]\n" + reasoning_details_text
                elif reasoning_details_text:
                    thoughts = reasoning_details_text
                else:
                    thoughts = raw_reasoning_text

                # Extract Token Counts
                reasoning_tokens = 0
                completion_tokens = 0
                prompt_tokens = 0

                if hasattr(response, 'usage') and response.usage:
                    usage = response.usage
                    completion_tokens = getattr(usage, 'completion_tokens', 0)
                    prompt_tokens = getattr(usage, 'prompt_tokens', 0)

                    if hasattr(usage, 'completion_tokens_details') and usage.completion_tokens_details:
                        details = usage.completion_tokens_details
                        # Catch Object format
                        if hasattr(details, 'reasoning_tokens'):
                            reasoning_tokens = details.reasoning_tokens
                        # Catch Dictionary format
                        elif isinstance(details, dict):
                            reasoning_tokens = details.get('reasoning_tokens', 0)
                        elif hasattr(details, '__dict__'):
                            reasoning_tokens = vars(details).get('reasoning_tokens', 0)

                provider_used = "UNKNOWN"
                if hasattr(response, 'model_extra') and response.model_extra:
                    provider_used = response.model_extra.get('provider', 'UNKNOWN')

                row[f"Diagnosis_{model['name']}"] = diag
                row[f"Likert_{model['name']}"] = likert
                row[f"Prompt_Tokens_{model['name']}"] = prompt_tokens
                row[f"Total_Tokens_Out_{model['name']}"] = completion_tokens
                row[f"Reasoning_Tokens_{model['name']}"] = reasoning_tokens
                row[f"Latency_{model['name']}"] = latency
                row[f"Provider_{model['name']}"] = provider_used
                row[f"Timestamp_UTC_{model['name']}"] = datetime.now(timezone.utc).isoformat()

                row[f"Reasoning_{model['name']}"] = thoughts
                row[f"Reasoning_Raw_{model['name']}"] = raw_reasoning_text
                row[f"Reasoning_Details_{model['name']}"] = reasoning_details_text

                row[f"Actual_Request_Extra_{model['name']}"] = json.dumps(
                    api_params.get("extra_body", None),
                    ensure_ascii=False,
                )

                row[f"Grok_Fallback_Used_{model['name']}"] = (
                    grok_fallback_used if model["name"] == "grok_4_20" else ""
                )

                row[f"OpenRouter_Response_Model_{model['name']}"] = getattr(response, "model", "")
                row[f"Usage_JSON_{model['name']}"] = json.dumps(
                    make_json_safe(getattr(response, "usage", None)),
                    ensure_ascii=False,
                )

                row[f"Raw_Response_{model['name']}"] = raw_answer[:2000]

                tps = round(completion_tokens / latency, 1) if latency > 0 else 0
                print(f" ✅ ({latency}s | {completion_tokens} out / {prompt_tokens} in | {tps} tok/sec)")

            except Exception as e:
                full_error = str(e)
                print(f" ❌ Failed! API Response: {full_error}")
                row[f"Diagnosis_{model['name']}"] = "API_ERROR"
                row[f"Likert_{model['name']}"] = "ERROR"
                row[f"Prompt_Tokens_{model['name']}"] = 0
                row[f"Total_Tokens_Out_{model['name']}"] = 0
                row[f"Reasoning_Tokens_{model['name']}"] = 0
                row[f"Latency_{model['name']}"] = 0
                row[f"Provider_{model['name']}"] = "ERROR"
                row[f"Timestamp_UTC_{model['name']}"] = datetime.now(timezone.utc).isoformat()
                row[f"Reasoning_{model['name']}"] = full_error
                row[f"Reasoning_Raw_{model['name']}"] = ""
                row[f"Reasoning_Details_{model['name']}"] = ""
                row[f"Actual_Request_Extra_{model['name']}"] = json.dumps(
                    api_params.get("extra_body", None) if api_params else None,
                    ensure_ascii=False,
                )
                row[f"Grok_Fallback_Used_{model['name']}"] = (
                    grok_fallback_used if model["name"] == "grok_4_20" else ""
                )
                row[f"OpenRouter_Response_Model_{model['name']}"] = ""
                row[f"Usage_JSON_{model['name']}"] = ""
                row[f"Raw_Response_{model['name']}"] = full_error[:2000]

        results.append(row)

        if idx % 5 == 0:
            pd.DataFrame(results).to_csv(output_csv.replace(".csv", "_BACKUP.csv"), index=False)

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"\n🏁 Complete! Data saved to {output_csv}")
    return df


In [ ]:
# ==========================================
# 6. EXECUTION
# ==========================================

master_images_folder = '/content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/RadLE v2 Master Data'
final_output_csv = '/content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/RadLE_v1.5_RSNA.csv'

# Set test_limit to process specific number or None for all.
# Recommended sequence: run test_limit=1 first; if syntax/API plumbing passes, run test_limit=5.
df_final = run_benchmark(master_images_folder, final_output_csv, test_limit=None)

print("\n📊 FINAL DATAFRAME PREVIEW:")
from IPython.display import display
display(df_final.head())


🚀 Processing 200 unique cases across 10 models...

[1/200] Case ID: 1 (1 images)
  -> gpt_5_5... ✅ (14.9s | 543 out / 1285 in | 36.4 tok/sec)
  -> claude_4_7_opus... ✅ (5.4s | 319 out / 1603 in | 59.1 tok/sec)
  -> gemini_3_1_pro... ✅ (8.1s | 634 out / 1413 in | 78.3 tok/sec)
  -> grok_4_20... ✅ (26.8s | 1852 out / 1231 in | 69.1 tok/sec)
  -> qwen_vl_max... ✅ (3.6s | 18 out / 1130 in | 5.0 tok/sec)
  -> gemma_4_31b... ✅ (25.7s | 692 out / 605 in | 26.9 tok/sec)
  -> llama_4_maverick... ✅ (3.4s | 79 out / 2196 in | 23.2 tok/sec)
  -> pixtral_large... ✅ (2.1s | 33 out / 3291 in | 15.7 tok/sec)
  -> glm_4_6v... ✅ (7.5s | 311 out / 1369 in | 41.5 tok/sec)
  -> nemotron_3_omni... ✅ (88.9s | 16384 out / 1134 in | 184.3 tok/sec)
[2/200] Case ID: 2 (1 images)
  -> gpt_5_5... ✅ (110.9s | 5725 out / 996 in | 51.6 tok/sec)
  -> claude_4_7_opus... ✅ (7.7s | 485 out / 1267 in | 63.0 tok/sec)
  -> gemini_3_1_pro... ✅ (6.9s | 435 out / 1409 in | 63.0 tok/sec)
  -> grok_4_20... ✅ (48.2s | 3766 out / 

In [ ]:
# ==========================================
# 7. SCORER'S VIEW & PIVOT (HUMAN-FRIENDLY)
# ==========================================

import os
import time
import base64
import pathlib
import pandas as pd
import json
import re
from collections import defaultdict
from openai import OpenAI
from google.colab import drive
from google.colab import userdata

# 1. Load the raw results
raw_csv = '/content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/RadLE_v1.5_RSNA.csv'
scorer_csv = raw_csv.replace(".csv", "_SCORER_VIEW.csv")

if os.path.exists(raw_csv):
    df = pd.read_csv(raw_csv)
    cols = df.columns.tolist()

    # 2. Define the Surgical Order
    # Identity first
    id_cols = ['Master_Case_ID', 'Associated_Images']

    # All Diagnoses together (sorted by model list order)
    diag_cols = [c for c in cols if 'Diagnosis_' in c]
    # All Likerts together
    likert_cols = [c for c in cols if 'Likert_' in c]

    # 3. Create the Human-Friendly Dataframe
    df_scorer = df[id_cols + diag_cols + likert_cols]

    # 4. Save as a separate file
    df_scorer.to_csv(scorer_csv, index=False)
    print(f"✅ Scorer version saved to: {scorer_csv}")

    # 5. THE CONSENSUS VIEW (Transposed for No-Scroll Review)
    # We look at Case 1 (or the last case) to see all 10 models side-by-side
    print("\n🧐 TRANSPOSED CONSENSUS VIEW (Case-by-Case):")

    # We filter only the Diagnoses and Likert columns for the display
    display_df = df[id_cols + diag_cols + likert_cols].set_index('Master_Case_ID')

    # Transposing makes the 10 models become rows—perfect for vertical reading
    display(display_df.T)

else:
    print("❌ ERROR: Raw CSV not found. Please run the benchmark first.")